# $$Prompt-Engineering$$

We have some prelimary knowledge about LLLMs already

- 
- what things / know how is needed for prompt engineering?



### Same prompts, different Results
 
- Supose that you are in an interview:

1. Question: **you are working on a in house LLM. you test the LLM by giving an input, the model gives good ouput. But in production the model gave different and bizzare outputs for the same prompt.**
    - giving the same prompt -> but recieving different results




**Next Token Prediction**
- LLMs don't look for answers like search engines
- Predict the next word ----
- Imagine the prompt: 
"The cat sat on the ..."
    - Mat: 80% prob
    - Floor: 15%
    - Pizza:5% 

**Temperature** - The chaos dial
- setting that flattens or sharpens the probability distribution
- Low Temperature: (0.1 to 0.3)
    - The model is greedy
    - always picks the highest probability words
- High Temperature: (0.7 - 1+)
    - The model takes more risks
    - It might pick the 15% or 5% options too.
    - Leads to more diverse, creative, and sometimes **hallucinatory** responses

In [5]:
import warnings
warnings.filterwarnings('ignore')

In [12]:
from transformers import pipeline

generator = pipeline("text-generation", model = "gpt2")

text = """
Write a 500-word blog post for B2B SaaS founders about using LinkedIN Ads to lower Customer Acquisition Cost(CAC). Use a punchy, authoriatative tone and include a bulleted list of 3 common mistakes"
"""

result = generator(text, temperature = 0.3, do_sample = True)
print(result[0]['generated_text'])

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 380.59it/s, Materializing param=transformer.wte.weight]             
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Write a 500-word blog post for B2B SaaS founders about using LinkedIN Ads to lower Customer Acquisition Cost(CAC). Use a punchy, authoriatative tone and include a bulleted list of 3 common mistakes"

"You can't just start with a single word, you have to start with a whole lot of words"

"You need to be able to use a lot of words to describe your product"

"You need to be able to use a lot of words to describe your product"

"You need to be able to use a lot of words to describe your product"

"You need to be able to use a lot of words to describe your product"

"You need to be able to use a lot of words to describe your product"

"You need to be able to use a lot of words to describe your product"

"You need to be able to use a lot of words to describe your product"

"You need to be able to use a lot of words to describe your product"

"You need to be able to use a lot of words to describe your product"

"You need to be able to use a lot of words to describe your product"

"You need t

### Importance of Context

- Many modern LLMS have huge context windows - 1M+ tokens - 
- Working memory of an LLM
- These can read several books at once ---> Reason why we are able to upload PDF files etc and get an analysis on them
- 
- The more we fill the context window with token ---> the more token you use:
    - the higher the latency gets
    - it takes longer to think
    - it costs more (if using an API)

- Imagine: 
- Context window is a whiteboard
- Every message yous send, and every response that AI gives, is written on a digital whiteboard
- LLMs re-read the entire whiteboard every it generates a new word 
- The whiteboard has physical limits (measured in token)
    - once it is full, the oldest information is erased to make room for new notes
    - Forgetting

**Lost in the Middle Phenomemon**
- Research points out that, LLMs are not equally good at reading the whole whiteboard
- They have a U-shaped attention span:
    - **High Attention**: on the information at the `beginning` (initial instructions)
    - **Low Attention**: on the information buried in the `middle` of a conversation or a 50 page pdf
    - **High Attention**: on the information at the very `end` (The most recent prompt)

- Prompt Engineering Tip: Keep the mission-critical instructions at the very botton of your prompt, right before the AI to execute

**Garbage IN, Garbage OUT (GIGO)**

- the quality of the output is directly limited by the quality and clarity of the input

- Biggest cause is: AI is a **"yes-man"**, and tries to fulfill any request, even a bad one

- Example:
    - Garbage IN: "Write a blog post about Marketing"
        - Result: "A generic, boring 300-word essay that sounds like Wikipedia entry from 2012
    - Quality IN: "Write a 500-word blog post for B2B SaaS founders about using LinkedIN Ads to lower Customer Acquisition Cost(CAC). Use a punchy, authoriatative tone and include a bulleted list of 3 common mistakes"
        - Result: A targeted, professional piece of content that actually provides value


To avoid the GIGO, a prompt need to pass the P.A.S. test:
- **Precision**: Use strong verbs and exact numbers (write 3 paragraphs instead of write a bit)
- **Audience**: Tell the AI who it is writing for. 
- **Structure**: Give the AI a skeleton to frame the outputs ("Use headers, a table of contents, and a concluding summary)

### Anatomy of a Prompt
- refers to the the building blocks of a prompt
- things used to get teh best and most accurate outputs
- 
1. **Role:** 
- Tell the AI who to act like
    - "Act like a senior Python developer..."
2. **Context:** 
- Provide background information or constraints to steer the model
    - "...working on a customer churn prediction project"
3. **Instruction/Directive/Task:**
- The specific action you want to take like Summarize, Write the code, Analyze, Write a rhyming poem
    - "Write a function to clean this dataset"
4. **Input Data:**
- The raw information to be processed
- A specific piece of content that you want processed like text, csv, ...
    - "[Paste a CSV data file, or CSV code snippet]"
5. **Output Format:** 
- The format of the the output result
- How you want the result to be delivered
    - "Return only the code block with comments"
    - 
    1. *Desired File Format*: in bullet points, as a JSON object, a text file, a word document, an image
    2. *Formatting*: Markdown, JSON, XML, CSV
    3. *Length*: Specific word-count, paragraph limits, one-sentence answers
    4. *Tone & Style*: Mimicking a specific author or brand voice, or use a certain style of language. 

**Example of Stuructured Prompts**

**Weak Prompt**

"Tell me how to build a machine learning model."

Result: You'll get a generic, 10-page essay that might not even apply to your specific tools.


**The Structured Prompt:**

`[Role]` Act as a Lead Data Scientist.
`[Context]` I am building a binary classification model using the Telco Customer Churn dataset in a Jupyter Notebook.
`[Task]` Help me perform feature engineering.
`[Instructions]` Focus on encoding categorical variables like 'Gender' and 'Contract' using One-Hot Encoding. Please ensure the code is compatible with Scikit-Learn.
`[Output]` Provide the Python code in a single block with a brief explanation of why we chose One-Hot Encoding over Label Encoding.

> **Tips: **
- *Be Direct*: Use `Do` instead of `Don't`
e.g. `Write a short summary` works better than `Don't write a long summary`
- *Use Few-Shot Prompting*: Provide 1 or 2 examples of the output you want
- *Demlimiters*: help in adding clarity. Use `(""")` , backticks `(```)`, or XML tags (`<data> ... </data>`)

### Prompting Gemini

- Benefits of using code windows to prompt LLMs:

1. **Unfiltered Control**: You can turn off the safety filters. The consumer site is much more restrictive
2. **System Instructions**: You can define a permamanent persona (e.g. Always respond in JSON, or markdown) that the model will not forget during chat. This is the *hidden prompt* that model always sees before your question, setting the rules for the entire session.

3. **Model Switching**: You can test the *Flash* model "speed" vs the *Pro* model instantly

4. **JSON Mode**: You can force the model to output data in a specific structure for your own apps.
5. **Temperature Slider**: You can manually dail down the creativity to 0 to get consistent and factual answers or increase it to 1+ for chaotic or risky answers

In [8]:
# pip install google-generativeai

In [9]:
import os
from dotenv import load_dotenv

import google.generativeai as genai

load_dotenv # to get the contents of a .env file into our system
# by default it will look into the same directory that we are working in first
api_key = os.getenv("google_api_key")

c:\Users\devid\Desktop\Nexperts Academy\AIML-nexperts3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\devid\AppData\Local\Temp\ipykernel_3952\968632520.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [29]:
# configuration
genai.configure(api_key = api_key)


model = genai.GenerativeModel(
    model_name= "gemini-2.5-flash-lite",
    system_instruction = "You are a grumpy librarian. Short and angry answers only"
)

response = model.generate_content(
    "How do I find a book on Sherlock Homes?", # user prompt

    generation_config = genai.types.GenerationConfig(
        candidate_count = 1 , # only give me 1 answer, don't waste any tokens on drafts or thinkning etc
        # stop_sequences = ["."] , # Cur the AI off as soon as it types a full stop
        max_output_tokens = 20, # Hard limit on lenght, to save our quota
        temperature = 0.3 # low --- keep answers factual only
    )
)

In [30]:
response

response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "candidates": [
        {
          "content": {
            "parts": [
              {
                "text": "Dewey Decimal. Fiction. 823."
              }
            ],
            "role": "model"
          },
          "finish_reason": "STOP",
          "index": 0
        }
      ],
      "usage_metadata": {
        "prompt_token_count": 22,
        "candidates_token_count": 11,
        "total_token_count": 33
      },
      "model_version": "gemini-2.5-flash-lite"
    }),
)

In [16]:

from google import genai as gen


client = gen.Client(api_key = api_key)

for m in client.models.list():
    if 'generateContent' in m.supported_actions:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025


Core Frameworks


RTF
CARE
COST

Advanced Prompting Strategies


Few Shot Prompting

Chain of Thought

Delimiters

Role Prompting

